# Figure generation — patient-facing XAI scoping review

Generates every figure in the manuscript and its supplement, writing a vector PDF (Type 42 embedded fonts) and a 600 dpi PNG preview into `figures/`.

Counts are derived from `data/appendix-d-extraction-matrix.xlsx` on every run — the workbook is the single source of truth. Editing a study, or adding one, changes the figures without anything else being touched. The CSV beside it is rewritten each run, so it cannot drift.

The one thing still coded outside the data is the PRISMA screening tallies (Figure 1), which describe records the matrix never held.

Run Setup first, then any figure cell independently.

## Setup

Loads the workbook, applies the Appendix B taxonomies, and defines the shared palette and plotting helpers. Every grouped variable is mapped through an explicit taxonomy: an unrecognised value raises here rather than being silently dropped, so a new study with a new category fails loudly instead of quietly vanishing from a figure.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SETUP — data load, standardised taxonomies, shared plotting helpers
# ═══════════════════════════════════════════════════════════════════════════════
# Every figure below derives its counts from the extraction matrix. Nothing is
# hard-coded except the PRISMA screening tallies (see Figure 1).
# Output: true vector PDF (editable text, embedded Type 42 fonts) + 600 dpi PNG.

import os
import re
from pathlib import Path

import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype']  = 42
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle, FancyBboxPatch
import numpy as np
import pandas as pd

DATA_PATH  = Path('data/appendix-d-extraction-matrix.xlsx')
CSV_PATH   = DATA_PATH.with_suffix('.csv')
OUTPUT_DIR = './figures'
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_excel(DATA_PATH)
df.columns = [str(c).strip() for c in df.columns]
df = df.dropna(how='all').reset_index(drop=True)
df = df.sort_values('Primary Author & Year').reset_index(drop=True)
N = len(df)

# The CSV is a derived artefact, rewritten on every run so it cannot drift from
# the workbook. Edit the xlsx only.
df.to_csv(CSV_PATH, index=False)
print(f'Loaded {N} studies from {DATA_PATH}; refreshed {CSV_PATH}')

# ───────────────────────────────────────────────────────────────────────────────
# Standardised taxonomies (Appendix B)
# ───────────────────────────────────────────────────────────────────────────────
# The matrix records each variable at its original granularity; the figures report
# the grouped categories. Mappings are keyed on the label before any parenthetical
# gloss. An unmapped value raises rather than being silently dropped, so a newly
# added study fails loudly here.

def _head(value):
    """Leading category label, before any parenthetical gloss."""
    return str(value).split('(')[0].strip()

# Grouped in the workbook itself (Clinical Domain (grouped), rules in Appendix B);
# listed here only so an unexpected label raises.
DOMAIN_TAXONOMY = {g: g for g in [
    'Internal Medicine', 'Oncology', 'Primary & Emergency Care',
    'Psychiatry', 'Surgery', 'Radiology']}

# Grouped per Table 1.
ARCHITECTURE_TAXONOMY = {
    'Simulated Algorithm': 'Simulated Algorithm',
    'Simulated AI Chatbot': 'Simulated Algorithm',
    'Simulated LLM': 'Simulated Algorithm',
    'Simulated Machine Learning': 'Simulated Algorithm',
    'Simulated Opaque Model': 'Simulated Algorithm',
    'Opaque': 'Black-Box + Post-Hoc',
    'Opaque ML Survival Model': 'Black-Box + Post-Hoc',
    'Optimised Artificial Neural Network': 'Black-Box + Post-Hoc',
    'Opaque LLM': 'LLM',
    'Modular Multi-modal LLM': 'LLM',
    'Hybrid Multi-modal': 'LLM',
    'Stacked Ensemble ML + RAG': 'LLM',
    'Transparent / Bayesian Network': 'Interpretable ML',
    'Transparent / Causal ML': 'Interpretable ML',
    'Transparent / Rule-based Expert System': 'Interpretable ML',
    'Transparent / Support Vector Machines': 'Interpretable ML',
}

DELIVERY_TAXONOMY = {
    'Web-based Application': 'Web Platform', 'Web-based Platform': 'Web Platform',
    'Web-based Simulation': 'Web Platform', 'Web-based Survey': 'Web Platform',
    'Adaptive Web/Mobile Platform': 'Web Platform',
    # Per Table 1: the video-simulated symptom checker was delivered through an
    # online experiment, whereas CareAssist-GPT was a proposed assistant, never built.
    'Video Simulation': 'Web Platform',
    'Conversational Interface': 'Conceptual Prototype',
    'Not applicable': 'Conceptual Prototype',
    'Conceptual Diagrams and Proposed Screener Questions': 'Conceptual Prototype',
    'Slides and Paper Mock-ups': 'Conceptual Prototype',
    'Mobile Application': 'Mobile Health App',
    'Paper/PDF Report': 'Static Report',
    'Tablet-based Application': 'In-Clinic App',
}

SCOPE_TAXONOMY = {'Local': 'Local', 'Local & Global': 'Local & Global', 'Global': 'Global'}

# Composite involvement labels are written "<earliest phase> & Evaluation", so the
# phase is the first listed.
PHASE_TAXONOMY = {
    'Needs Assessment': 'Needs Assessment', 'Co-design': 'Co-design',
    'Evaluation': 'Summative Evaluation', 'Implementation': 'Clinical Implementation',
}

def _classify_design(value):
    """RCTs are separated from other randomised designs before the generic prefix test."""
    if value.startswith('Qualitative'):                   return 'Qualitative'
    if value.startswith('Randomised controlled trial'):   return 'Randomised Controlled Trial'
    if value.startswith('Randomised'):                    return 'Randomised Experiment'
    if value.startswith('Mixed-methods'):                 return 'Mixed Methods'
    return 'Observational Study'

def _classify_modality(value):
    """Hybrid covers any text+visual combination; Visual is text-free output."""
    if value.startswith(('Hybrid', 'Multimodal')):        return 'Hybrid'
    if 'extual' in value or value.startswith('Conversational'): return 'Textual'
    return 'Visual'

def _grouped(column, taxonomy, label):
    keys = df[column].map(_head)
    unmapped = sorted(set(keys) - set(taxonomy))
    if unmapped:
        raise KeyError(f'{label}: no taxonomy entry for {unmapped!r}. Add it above.')
    return keys.map(taxonomy)

def cite(value):
    """'Alam, L. & Mueller, S. (2021)' -> 'Alam & Mueller (2021)'."""
    body, year = re.match(r'(.+?)\s*\((\d{4})\)\s*$', str(value)).groups()
    surname = lambda p: re.split(r',| et al', p)[0].strip()
    parts = [p.strip() for p in body.split('&')]
    if len(parts) == 2:   name = f'{surname(parts[0])} & {surname(parts[1])}'
    elif 'et al' in body: name = f'{surname(body)} et al.'
    else:                 name = surname(body)
    return f'{name} ({year})'

CODED = pd.DataFrame({
    'study':        df['Primary Author & Year'],
    'domain':       _grouped('Clinical Domain (grouped)', DOMAIN_TAXONOMY, 'Clinical Domain (grouped)'),
    'task':         df['Clinical Task'].map(_head),
    'design':       df['Study Design & Methodology'].map(_classify_design),
    'architecture': _grouped('AI Architecture', ARCHITECTURE_TAXONOMY, 'AI Architecture'),
    'modality':     df['Explanation Modality'].map(_classify_modality),
    'scope':        _grouped('Explanation Scope', SCOPE_TAXONOMY, 'Explanation Scope'),
    'delivery':     _grouped('Delivery Format', DELIVERY_TAXONOMY, 'Delivery Format'),
    'phase':        df['Patient Involvement Phase'].str.split('&').str[0].str.strip().map(PHASE_TAXONOMY),
    'country':      df['Country of Origin'].map(_head),
    'year':         df['Primary Author & Year'].str.extract(r'\((\d{4})\)')[0].astype(int),
})
CODED['cite'] = CODED['study'].map(cite)
assert not CODED.isna().any().any(), f'unmapped values:\n{CODED[CODED.isna().any(axis=1)]}'

# Declared order breaks ties so bars stay stable when counts are equal.
CATEGORY_ORDER = {
    'domain':       ['Internal Medicine', 'Oncology', 'Primary & Emergency Care',
                     'Psychiatry', 'Surgery', 'Radiology'],
    'task':         ['Diagnosis', 'Triage', 'Prognosis', 'Self-management',
                     'Screening', 'Treatment Selection'],
    'design':       ['Qualitative', 'Randomised Experiment', 'Observational Study',
                     'Randomised Controlled Trial', 'Mixed Methods'],
    'architecture': ['Simulated Algorithm', 'Black-Box + Post-Hoc', 'LLM', 'Interpretable ML'],
    'modality':     ['Textual', 'Hybrid', 'Visual'],
    'scope':        ['Local', 'Local & Global', 'Global'],
    'delivery':     ['Web Platform', 'Conceptual Prototype', 'Mobile Health App',
                     'Static Report', 'In-Clinic App'],
    'phase':        ['Summative Evaluation', 'Needs Assessment', 'Co-design',
                     'Clinical Implementation'],
    'subjective':   ['Validated Psychometric Scales', 'Interviews & Focus Groups',
                     'Unvalidated Questionnaires'],
    'objective':    ['Not Evaluated', 'Patient Comprehension & Accuracy',
                     'AI Algorithm Performance'],
    'behavioural':  ['Not Evaluated', 'Experimental Setting', 'Real-World Clinical'],
}

# Line breaks for the narrow multi-panel axes in Figure 2.
WRAPPED = {
    'Internal Medicine': 'Internal\nMedicine', 'Primary & Emergency Care': 'Primary &\nEmergency Care',
    'Self-management': 'Self-\nManagement', 'Treatment Selection': 'Treatment\nSelection',
    'Randomised Experiment': 'Randomised\nExperiment', 'Observational Study': 'Observational\nStudy',
    'Randomised Controlled Trial': 'Randomised\nControlled Trial',
    'Simulated Algorithm': 'Simulated\nAlgorithm', 'Black-Box + Post-Hoc': 'Black-Box +\nPost-Hoc',
    'Interpretable ML': 'Interpretable\nML', 'Summative Evaluation': 'Summative\nEvaluation',
    'Needs Assessment': 'Needs\nAssessment', 'Clinical Implementation': 'Clinical\nImplementation',
    'Conceptual Prototype': 'Conceptual\nPrototype', 'Mobile Health App': 'Mobile\nHealth App',
}

def grouped_counts(key, wrap=False):
    """Labels and counts for one coded variable, ordered by descending count."""
    counts = CODED[key].value_counts()
    order = [c for c in CATEGORY_ORDER[key] if c in counts.index]
    order += sorted(set(counts.index) - set(order))
    order.sort(key=lambda c: -counts[c])
    labels = [WRAPPED.get(c, c) if wrap else c for c in order]
    return labels, [int(counts[c]) for c in order]



# ───────────────────────────────────────────────────────────────────────────────
# Metric tiers and trust calibration (coded in the workbook)
# ───────────────────────────────────────────────────────────────────────────────
# Each metric cell ends with its tier in square brackets, e.g. "... [Not Evaluated]".
# Trust Calibration names its matrix cell in parentheses. Missing or unrecognised
# codes raise, so an incompletely coded study cannot slip into a figure.

def _tier(column, key):
    tags = df[column].astype(str).str.extract(r'\[([^\]]+)\]\s*\.?\s*$')[0].str.strip()
    untagged = CODED.loc[tags.isna(), 'cite'].tolist()
    if untagged:
        raise KeyError(f'{column}: no [tier] tag for {untagged}')
    unknown = sorted(set(tags) - set(CATEGORY_ORDER[key]))
    if unknown:
        raise KeyError(f'{column}: unrecognised tier {unknown!r}. Add it to CATEGORY_ORDER.')
    return tags

CODED['subjective']  = _tier('Subjective Metrics', 'subjective')
CODED['objective']   = _tier('Objective Metrics', 'objective')
CODED['behavioural'] = _tier('Behavioural Metrics', 'behavioural')

TRUST_CELL_TAXONOMY = {
    'Concordant (Appropriate Reliance)':   ('HIGH', 'accurate'),
    'Discordant (Over-Reliance)':          ('HIGH', 'inaccurate'),
    'Unverified (Unverified Reliance)':    ('HIGH', 'unknown'),
    'Discordant (Under-Reliance)':         ('LOW',  'accurate'),
    'Concordant (Appropriate Scepticism)': ('LOW',  'inaccurate'),
    'Unverified (Unverified Scepticism)':  ('LOW',  'unknown'),
}
_trust = df['Trust Calibration'].astype(str).str.strip()
_unknown = sorted(set(_trust) - set(TRUST_CELL_TAXONOMY))
if _unknown:
    raise KeyError(f'Trust Calibration: unrecognised value {_unknown!r}')
CODED['trust_level']       = _trust.map(lambda v: TRUST_CELL_TAXONOMY[v][0])
CODED['model_performance'] = _trust.map(lambda v: TRUST_CELL_TAXONOMY[v][1])

def testing_n(value):
    """First reported testing cohort; a usability sub-sample is shown in brackets."""
    if str(value).strip().startswith('Not'):
        return 'Not Reported'
    nums = re.findall(r'[Nn]\s*=\s*([\d,]+)', str(value))
    if not nums:
        return 'Not Reported'
    if re.search(r'sub-se|sub-sam', str(value)) and len(nums) > 1:
        return f'{nums[0]} ({nums[1]})*'
    return nums[0]

CODED['testing_n'] = df['Testing N'].map(testing_n)

# ───────────────────────────────────────────────────────────────────────────────
# House style and shared plotting helpers
# ───────────────────────────────────────────────────────────────────────────────
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']

BAR_MAIN   = '#2b5c8f'
BAR_PEAK   = '#1a365d'
BAR_PART   = '#7ca3cd'
BAR_ABSENT = '#94a3b8'
BG_AX      = '#f8fafc'
SPINE_COL  = '#cbd5e1'
GRID_COL   = '#e2e8f0'
INK        = '#111111'
T_GREY     = '#64748b'

CONCORD_BORDER = '#1a365d'
CONCORD_FILL   = '#ffffff'
DISCORD_FILL   = '#fbe4e4'
DISCORD_TEXT   = '#b91c1c'
UNVER_FILL     = '#eef3fb'
UNVER_BORDER   = '#a9bdd6'

FS_AX      = 10.0
FS_VAL     = 10.0
FS_LAB     = 11.0
FS_PAN     = 12.0
FS_SUP     = 14.0
def _panel_title(ax, letter, title_text):
    ax.set_title(f"{letter}   {title_text}", loc='left', fontsize=FS_PAN,
                 fontweight='bold', pad=48, color=INK)

def _suptitle(fig, text, y_pos=0.96):
    fig.suptitle(text, fontsize=FS_SUP, fontweight='bold', color=INK, y=y_pos)

def _save(fig, filename):
    pdf_path = os.path.join(OUTPUT_DIR, f"{filename}.pdf")
    png_path = os.path.join(OUTPUT_DIR, f"{filename}.png")
    # CreationDate=None keeps the PDF byte-identical between runs, so committed
    # figures only change when the figure does.
    fig.savefig(pdf_path, format='pdf', bbox_inches='tight', facecolor='white',
                metadata={'CreationDate': None})
    fig.savefig(png_path, format='png', dpi=600, bbox_inches='tight', facecolor='white')
    print(f"   ✅ Saved: {pdf_path}")
    print(f"   ✅ Saved: {png_path}")
    plt.close(fig)

def _horiz_bar(ax, categories, values, letter, title_text,
               colors, xpad=2.0, val_fmt=None):
    ax.set_facecolor(BG_AX)
    y_pos = np.arange(len(categories))
    hbars = ax.barh(y_pos, values, color=colors, height=0.58, zorder=3, edgecolor='white', linewidth=0.5)
    for bar, v in zip(hbars, values):
        w = bar.get_width()
        label = val_fmt(v) if val_fmt else f'n = {int(w)}'
        ax.annotate(label, xy=(w, bar.get_y() + bar.get_height() / 2),
                    xytext=(6, 0), textcoords='offset points', va='center', ha='left',
                    fontsize=FS_VAL, fontweight='bold', color=INK)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(categories, fontsize=FS_AX, color=INK)
    ax.invert_yaxis()
    ax.set_xlabel('Number of Studies', fontsize=FS_LAB, color=INK, labelpad=5)
    ax.set_xlim(0, max(values) + xpad)
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    for spine in ['top', 'right']: ax.spines[spine].set_visible(False)
    ax.spines['left'].set_color(SPINE_COL); ax.spines['bottom'].set_color(SPINE_COL)
    ax.grid(axis='x', linestyle=':', alpha=0.55, color=GRID_COL)
    ax.grid(axis='y', visible=False)
    ax.tick_params(axis='y', length=0, pad=5)
    ax.tick_params(axis='x', labelsize=9.5)
    _panel_title(ax, letter, title_text)

# ───────────────────────────────────────────────────────────────────────────────
# Table 1 and the Trust Calibration cells
# ───────────────────────────────────────────────────────────────────────────────
# Shared by the figure that renders them and the cell that writes the CSVs, so the
# two cannot disagree.

# Table 1 uses fuller category names than the figure axes.
TABLE_NAMES = {
    'Qualitative': 'Qualitative Study',
    'Black-Box + Post-Hoc': 'Black-Box ML', 'LLM': 'Large Language Model',
    'Interpretable ML': 'Interpretable Machine Learning', 'Self-management': 'Self-Management',
    'Summative Evaluation': 'Summative Testing', 'The Netherlands': 'Netherlands',
}
TABLE1_COLUMNS = ['Author (Year)', 'Country', 'Clinical Domain', 'Clinical Task', 'Study Design',
                  'Testing (n)', 'AI Architecture', 'Explanation Modality', 'Delivery Format',
                  'Patient Involvement Phase']

def table1():
    """Characteristics of included studies, one row per study."""
    name = lambda v: TABLE_NAMES.get(v, v)
    rows = [[r.cite, name(r.country), name(r.domain), name(r.task), name(r.design), r.testing_n,
             name(r.architecture), r.modality, r.delivery, name(r.phase)]
            for r in CODED.itertuples()]
    return pd.DataFrame(rows, columns=TABLE1_COLUMNS)

TRUST_CELL_NAMES = {
    ('HIGH', 'accurate'):   'Appropriate Reliance',   ('HIGH', 'inaccurate'): 'Over-Reliance',
    ('HIGH', 'unknown'):    'Unverified Reliance',    ('LOW',  'accurate'):   'Under-Reliance',
    ('LOW',  'inaccurate'): 'Appropriate Scepticism', ('LOW',  'unknown'):    'Unverified Scepticism',
}
TRUST_CELL_KIND = {
    ('HIGH', 'accurate'):   'concordant', ('LOW',  'inaccurate'): 'concordant',
    ('HIGH', 'inaccurate'): 'discordant', ('LOW',  'accurate'):   'discordant',
    ('HIGH', 'unknown'):    'unverified', ('LOW',  'unknown'):    'unverified',
}

def trust_members(key):
    """Studies falling in one cell of the Trust Calibration Matrix."""
    hit = (CODED['trust_level'] == key[0]) & (CODED['model_performance'] == key[1])
    return sorted(CODED[hit]['cite'])

VARIABLE_LABELS = {
    'domain': 'Clinical domain', 'task': 'Clinical task', 'design': 'Study design',
    'architecture': 'AI architecture', 'modality': 'Explanation modality',
    'scope': 'Explanation scope', 'delivery': 'Delivery format',
    'phase': 'Patient involvement phase', 'subjective': 'Subjective metrics',
    'objective': 'Objective metrics', 'behavioural': 'Behavioural metrics',
}

## Figure 1 — PRISMA-ScR flow diagram

Study selection across the dual-pathway search, database records and citation tracking, narrowing to the included studies.

The screening tallies are the one set of numbers still written into the code: the extraction matrix holds only the studies that survived screening, so records excluded along the way have no row to derive from. Each exclusion reason is entered once; the totals and the included count per arm are computed from them, and the cell stops if they do not add up to the studies in the workbook. Box positions are laid out by hand, so edits mean moving explicit x/y values.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 1 (PRISMA FLOW DIAGRAM)
# ═══════════════════════════════════════════════════════════════════════════════
# Screening tallies come from the Covidence audit trail (Appendix C); the
# extraction matrix holds only the included studies, so these cannot be derived.
# Enter each exclusion reason once. Totals and the included count per arm are
# computed from them, and must add up to the studies in the workbook.

DB_ASSESSED  = 123   # database arm: full-text reports assessed
DB_SECONDARY = 45    # secondary reviews routed to citation tracking
DB_EXCLUDED = {      # primary research excluded at full text
    'No human evaluation': 18,
    'Conference abstracts': 15,
    'Educational conversational models': 8,   # incl. Maywald 2025, removed after co-author review
    'Full-text unavailable': 6,
    'Clinician-facing XAI': 5,
    'Trial registries': 4,
    'Non-English': 2,
    'Text simplification': 2,
    'Attitude surveys': 1,
    'Black-box risk calculator': 1,
}
CT_ASSESSED = 176    # citation-tracking arm: full-text reports assessed
CT_EXCLUDED = {
    'Black-box wearables': 44,
    'Black-box diagnostics': 31,
    'Clinician-facing XAI': 26,
    'Educational models': 25,
    'Attitude surveys': 16,
    'Text simplification': 14,
    'Risk calculators / Other': 16,
}

DB_INCLUDED = DB_ASSESSED - DB_SECONDARY - sum(DB_EXCLUDED.values())
CT_INCLUDED = CT_ASSESSED - sum(CT_EXCLUDED.values())
assert DB_INCLUDED + CT_INCLUDED == N, (
    f'PRISMA arms include {DB_INCLUDED} + {CT_INCLUDED} = {DB_INCLUDED + CT_INCLUDED} '
    f'studies but the workbook holds {N}: update the exclusion tallies')

def _excl_text(reasons, secondary=None):
    primary = sum(reasons.values())
    if secondary is None:
        return (f'Reports excluded (n = {primary}):\n'
                + '\n'.join(f'• {k} (n = {v})' for k, v in reasons.items()))
    return (f'Reports excluded (n = {primary + secondary}):\n'
            f'• Secondary reviews routed to citation tracking (n = {secondary})\n'
            f'• Primary research excluded (n = {primary}):\n'
            + '\n'.join(f'  - {k} (n = {v})' for k, v in reasons.items()))

def make_figure_1_prisma():
    print("⏳ Generating Figure 1 (PRISMA flow diagram)...")
    fig, ax = plt.subplots(figsize=(19.0, 15.0))
    ax.set_xlim(0, 200)
    ax.set_ylim(0, 110)
    ax.axis('off')

    BAND_LIGHT = '#f1f5f9'
    BAND_WHITE = '#ffffff'

    def band(y0, y1, color, label):
        ax.add_patch(Rectangle((0, y0), 200, y1 - y0, facecolor=color, edgecolor='none', zorder=0))
        if label:
            ax.text(1.5, y1 - 2.5, label, fontsize=12.0, fontweight='bold', color='#334155', va='top', ha='left', zorder=6, linespacing=1.35)

    band(0, 20, BAND_LIGHT, f'INCLUDED\n(n = {N})')
    band(20, 78, BAND_WHITE, 'SCREENING')
    band(78, 110, BAND_LIGHT, 'IDENTIFICATION')
    for y in [20, 78]:
        ax.plot([0, 200], [y, y], color='#cbd5e1', lw=1.2, zorder=1)

    def add_box(xc, yc, w, h, text, fontsize=11.0, facecolor='white',
                edgecolor='#334155', weight='normal', lw=1.1, zorder=3, ha='center'):
        ax.add_patch(Rectangle((xc - w / 2, yc - h / 2), w, h,
                                facecolor=facecolor, edgecolor=edgecolor,
                                linewidth=lw, zorder=zorder))
        tx = (xc - w / 2 + 2.5) if ha == 'left' else xc
        ax.text(tx, yc, text, fontsize=fontsize, color=INK, ha=ha,
                 va='center', linespacing=1.35, fontweight=weight, zorder=zorder + 1)

    def add_arrow(x1, y1, x2, y2, lw=1.3):
        ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                     arrowprops=dict(arrowstyle='-|>', color='#334155', lw=lw,
                                     shrinkA=0, shrinkB=0, mutation_scale=15),
                     zorder=2)

    def add_header(x0, x1, yc, h, text):
        ax.add_patch(FancyBboxPatch((x0, yc - h / 2), x1 - x0, h,
                                     boxstyle="round,pad=0.35,rounding_size=0.5",
                                     facecolor='#dbe5f1', edgecolor='#94b0d1',
                                     linewidth=1.1, zorder=3))
        ax.text((x0 + x1) / 2, yc, text, fontsize=13.0, fontweight='bold', color=INK,
                 ha='center', va='center', zorder=4)

    # Core meticulously balanced coordinates
    C1, W1 = 28, 38
    EX1, W_EX1 = 78, 46
    C3, W3 = 128, 38
    EX2, W_EX2 = 176, 44

    add_header(7, 102, 99.0, 5.0, 'Databases & Registers')
    add_header(107, 199, 99.0, 5.0, 'Citation Tracking')

    yA = 89
    add_box(C1, yA, W1, 10, 'Records identified from:\nPubMed, MEDLINE, Embase, Scopus\n(n = 1,234)', fontsize=10.5)
    add_box(EX1, yA, W_EX1, 10, 'Records removed before screening:\nDuplicate records removed\n(n = 604)', fontsize=10.0)
    add_box(C3, yA, W3, 10, 'Records identified via citation\ntracking & manual search\n(n = 808)', fontsize=10.5)
    add_arrow(C1 + W1/2, yA, EX1 - W_EX1/2, yA)

    yB = 68
    add_box(C1, yB, W1, 10, 'Records screened\n(n = 630)', fontsize=11.0)
    add_box(EX1, yB, W_EX1, 10, 'Records excluded at title\nand abstract stage\n(n = 507)', fontsize=10.0)
    add_box(C3, yB, W3, 10, 'Leads screened\n(n = 808)', fontsize=11.0)
    add_box(EX2, yB, W_EX2, 10, 'Leads excluded at title\nand abstract stage\n(n = 632)', fontsize=10.0)

    add_arrow(C1, yA - 5, C1, yB + 5)
    add_arrow(C1 + W1/2, yB, EX1 - W_EX1/2, yB)
    add_arrow(C3, yA - 5, C3, yB + 5)
    add_arrow(C3 + W3/2, yB, EX2 - W_EX2/2, yB)

    yC = 49
    add_box(C1, yC, W1, 10, f'Reports sought for retrieval\n(n = {DB_ASSESSED})', fontsize=10.5)
    add_box(C3, yC, W3, 10, f'Reports sought for retrieval\n(n = {CT_ASSESSED})', fontsize=10.5)
    add_arrow(C1, yB - 5, C1, yC + 5)
    add_arrow(C3, yB - 5, C3, yC + 5)

    yE = 33
    add_box(C1, yE, W1, 8, f'Reports assessed (n = {DB_ASSESSED})', fontsize=10.5)
    add_box(C3, yE, W3, 8, f'Reports assessed (n = {CT_ASSESSED})', fontsize=10.5)
    add_arrow(C1, yC - 5, C1, yE + 4)
    add_arrow(C3, yC - 5, C3, yE + 4)

    yD = 33
    box_h = 24
    add_box(EX1, yD, W_EX1, box_h,
            _excl_text(DB_EXCLUDED, DB_SECONDARY), fontsize=8.5, ha='left')

    add_box(EX2, yD, W_EX2, box_h,
            _excl_text(CT_EXCLUDED), fontsize=9.5, ha='left')

    add_arrow(C1 + W1/2, yE, EX1 - W_EX1/2, yD, lw=1.4)
    add_arrow(C3 + W3/2, yE, EX2 - W_EX2/2, yD, lw=1.4)

    yF = 10
    add_box((C1+C3)/2, yF, 120, 12,
            f'Studies included in review (n = {N})\n'
            f'• Included via database searches (n = {DB_INCLUDED})\n'
            f'• Included via citation tracking (n = {CT_INCLUDED})',
            fontsize=11.5, weight='bold', facecolor='#eef2f7', edgecolor='#94b0d1')

    add_arrow(C1, yE - 4, C1, yF + 6)
    add_arrow(C3, yE - 4, C3, yF + 6)

    fig.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.01)
    _save(fig, 'figure-1-prisma-flow')

make_figure_1_prisma()

## Figure 2 — Clinical, methodological and technical landscape

Five panels over the same studies: clinical domain, clinical task, study design, AI architecture, and earliest phase of patient involvement. Clinical domain is read from the workbook's grouped column; the other panels group raw values through the Appendix B taxonomies in Setup.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 2 (LANDSCAPE)
# ═══════════════════════════════════════════════════════════════════════════════
def make_figure_2_landscape():
    print("⏳ Generating Figure 2 (landscape)...")
    fig, axes = plt.subplots(1, 5, figsize=(23.0, 5.5), sharey=False)
    domains,    domain_counts    = grouped_counts('domain', wrap=True)
    tasks,      task_counts      = grouped_counts('task', wrap=True)
    designs,    design_counts    = grouped_counts('design', wrap=True)
    architects, architect_counts = grouped_counts('architecture', wrap=True)
    phases,     phase_counts     = grouped_counts('phase', wrap=True)

    _horiz_bar(axes[0], domains, domain_counts, 'A', 'Clinical Domain', colors=[BAR_MAIN]*len(domains), xpad=2.0)
    _horiz_bar(axes[1], tasks, task_counts, 'B', 'Clinical Task', colors=[BAR_MAIN]*len(tasks), xpad=2.0)
    _horiz_bar(axes[2], designs, design_counts, 'C', 'Study Design', colors=[BAR_MAIN]*len(designs), xpad=2.0)
    _horiz_bar(axes[3], architects, architect_counts, 'D', 'AI Architecture', colors=[BAR_MAIN]*len(architects), xpad=2.0)
    _horiz_bar(axes[4], phases, phase_counts, 'E', 'Involvement Phase', colors=[BAR_MAIN]*len(phases), xpad=3.5)

    fig.subplots_adjust(left=0.03, right=0.99, top=0.65, bottom=0.15, wspace=0.38)
    _suptitle(fig, 'Clinical, Methodological and Technical Landscape', y_pos=0.92)
    _save(fig, 'figure-2-landscape')

make_figure_2_landscape()

## Figure 3 — Evaluation and outcome metrics

Outcome measures split into subjective, objective and behavioural tiers, with grey marking studies reporting nothing in that domain. Each study's tier is the bracketed tag at the end of its metric cell in the workbook.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 3 (EVALUATION METRICS)
# ═══════════════════════════════════════════════════════════════════════════════
def make_figure_3_evaluation_metrics():
    print("⏳ Generating Figure 3 (evaluation metrics)...")
    subj_labels, subj_n = grouped_counts('subjective')
    obj_labels,  obj_n  = grouped_counts('objective')
    beh_labels,  beh_n  = grouped_counts('behavioural')
    # Grey marks the studies reporting nothing in that domain, wherever it sorts.
    absent = lambda labels: [BAR_ABSENT if l == 'Not Evaluated' else BAR_MAIN for l in labels]
    fig, axes = plt.subplots(3, 1, figsize=(14.0, 11.5), gridspec_kw={'hspace': 0.85})
    _horiz_bar(axes[0], subj_labels, subj_n, 'A', 'Subjective Metrics', colors=[BAR_MAIN]*len(subj_n), xpad=1.5)
    _horiz_bar(axes[1], obj_labels, obj_n, 'B', 'Objective Metrics', colors=absent(obj_labels), xpad=1.5)
    _horiz_bar(axes[2], beh_labels, beh_n, 'C', 'Behavioural Metrics', colors=absent(beh_labels), xpad=1.5)
    ne = mpatches.Patch(facecolor=BAR_ABSENT, edgecolor=SPINE_COL, linewidth=0.7, label='Studies not reporting data in this domain')
    fig.subplots_adjust(left=0.16, right=0.98, top=0.88, bottom=0.10)
    fig.legend(handles=[ne], loc='lower center', bbox_to_anchor=(0.5, 0.02), frameon=True, fontsize=11)
    _suptitle(fig, 'Evaluation and Outcome Metrics', y_pos=0.96)
    _save(fig, 'figure-3-evaluation-metrics')

make_figure_3_evaluation_metrics()

## Figure 4 — Trust Calibration Matrix

The review's central construct: subjective patient trust (rows) against verified model accuracy (columns). Cell membership comes from the Trust Calibration column, whose parenthetical names the matrix cell directly.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 4 (TRUST CALIBRATION MATRIX)
# ═══════════════════════════════════════════════════════════════════════════════
def make_figure_4_trust_calibration():
    print("⏳ Generating Figure 4 (Trust Calibration Matrix)...")

    cells = {}
    for key, title in TRUST_CELL_NAMES.items():
        members = trust_members(key)
        cells[key] = dict(title=title, n=len(members), pct=round(len(members) / N * 100),
                          kind=TRUST_CELL_KIND[key], studies=members)

    style_map = {
        'concordant': dict(facecolor=CONCORD_FILL, edgecolor=CONCORD_BORDER, lw=2.2, textcolor=INK),
        'discordant': dict(facecolor=DISCORD_FILL, edgecolor='#e2b4b4', lw=1.0, textcolor=DISCORD_TEXT),
        'unverified': dict(facecolor=UNVER_FILL, edgecolor=UNVER_BORDER, lw=1.0, textcolor=INK),
    }

    col_keys = ['accurate', 'inaccurate', 'unknown']
    col_titles = ['AI Model Accurate', 'AI Model Inaccurate', 'AI Model Unknown']
    row_keys = ['HIGH', 'LOW']
    row_titles = ['HIGH\nPatient Trust', 'LOW\nPatient Trust']

    fig, ax = plt.subplots(figsize=(14.0, 8.5))
    ax.set_xlim(0, 115)
    ax.set_ylim(0, 100)
    ax.axis('off')

    LABEL_C, LABEL_W = 8, 14
    COL_X = {'accurate': 32, 'inaccurate': 65, 'unknown': 98}
    COL_W = 28
    HEADER_Y, HEADER_H = 88, 9
    ROW_Y = {'HIGH': 62, 'LOW': 22}
    ROW_H = 32

    for k, title in zip(col_keys, col_titles):
        ax.add_patch(Rectangle((COL_X[k] - COL_W / 2, HEADER_Y - HEADER_H / 2), COL_W, HEADER_H,
                                facecolor='#eef2f7', edgecolor='#c9d3de', lw=0.8, zorder=2))
        ax.text(COL_X[k], HEADER_Y, title, fontsize=11, fontweight='bold', ha='center', va='center', color=INK, zorder=3)

    for rk, rtitle in zip(row_keys, row_titles):
        ax.add_patch(Rectangle((LABEL_C - LABEL_W / 2, ROW_Y[rk] - ROW_H / 2), LABEL_W, ROW_H,
                                facecolor='#eef2f7', edgecolor='#c9d3de', lw=0.8, zorder=2))
        ax.text(LABEL_C, ROW_Y[rk], rtitle, fontsize=10.5, fontweight='bold', ha='center', va='center', color=INK, zorder=3)

    for rk in row_keys:
        for ck in col_keys:
            d = cells[(rk, ck)]
            s = style_map[d['kind']]
            xc, yc = COL_X[ck], ROW_Y[rk]
            ax.add_patch(Rectangle((xc - COL_W / 2, yc - ROW_H / 2), COL_W, ROW_H,
                                    facecolor=s['facecolor'], edgecolor=s['edgecolor'],
                                    linewidth=s['lw'], zorder=2))
            top_y = yc + ROW_H / 2 - 3.2
            ax.text(xc - COL_W / 2 + 1.2, top_y, d['title'], fontsize=10, fontweight='bold',
                    color=INK, ha='left', va='top', zorder=3)
            ax.text(xc - COL_W / 2 + 1.2, top_y - 4.2, f"n = {d['n']} ({d['pct']}%)", fontsize=9.5,
                    fontweight='bold', color=s['textcolor'], ha='left', va='top', zorder=3)
            rule_y = top_y - 6.4
            ax.plot([xc - COL_W / 2 + 1.2, xc + COL_W / 2 - 1.2], [rule_y, rule_y],
                    color=s['edgecolor'], lw=0.8, alpha=0.7, zorder=3)
            study_text = '\n'.join(d['studies'])
            ax.text(xc - COL_W / 2 + 1.2, rule_y - 1.8, study_text, fontsize=7.6,
                    color=T_GREY, ha='left', va='top', linespacing=1.6, zorder=3)

    legend_items = [
        ('Concordant', dict(facecolor=CONCORD_FILL, edgecolor=CONCORD_BORDER, lw=2.2)),
        ('Discordant', dict(facecolor=DISCORD_FILL, edgecolor='#e2b4b4', lw=1.0)),
        ('Unverified', dict(facecolor=UNVER_FILL, edgecolor=UNVER_BORDER, lw=1.0)),
    ]
    lx = 30
    ax.text(15, 98.5, 'Category:', fontsize=10.5, fontweight='bold', ha='left', va='center', color=INK)
    for label, sty in legend_items:
        ax.add_patch(Rectangle((lx, 97), 3, 3, **sty, zorder=3))
        ax.text(lx + 4.2, 98.5, label, fontsize=10, ha='left', va='center', color=INK)
        lx += 20

    ax.set_title('Trust Calibration Matrix', fontsize=FS_SUP, fontweight='bold', color=INK, pad=20, loc='center')
    fig.subplots_adjust(left=0.02, right=0.98, top=0.92, bottom=0.05)
    _save(fig, 'figure-4-trust-calibration-matrix')

make_figure_4_trust_calibration()

## Table 1 — Characteristics of included studies

One row per study, built straight from the matrix. Testing N takes the first reported cohort; where a usability sub-sample was also reported it follows in brackets and is footnoted.

> This cell renders the table as an image, which is how it was submitted. BMJ typesets tables from text, so `tables/table-1-study-characteristics.csv` is the version to submit — the image is a visual proof only.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TABLE 1 (STUDY CHARACTERISTICS)
# ═══════════════════════════════════════════════════════════════════════════════
def make_table_1_characteristics():
    print("⏳ Generating Table 1 (study characteristics)...")
    columns = TABLE1_COLUMNS
    data = table1().values.tolist()

    # ---------------------------------------------------------
    # UPDATED COLUMN WIDTHS:
    # Patient Phase down from 0.17 -> 0.13
    # AI Architecture up from 0.11 -> 0.13
    # Explanation Modality up from 0.08 -> 0.10
    # Overall sum remains 1.0
    # ---------------------------------------------------------
    col_widths = [0.13, 0.07, 0.11, 0.09, 0.10, 0.05, 0.13, 0.10, 0.09, 0.13]

    fig, ax = plt.subplots(figsize=(20.0, 9.2))
    ax.axis('off')
    tbl = ax.table(cellText=data, colLabels=columns, cellLoc='left', colWidths=col_widths, loc='upper center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9.5)
    tbl.scale(1, 1.85)

    HEADER_BG = '#dbe5f1'
    ROW_ALT   = '#f0f4f9'

    for (r, c), cell in tbl.get_celld().items():
        cell.set_edgecolor('#cbd5e1')
        cell.set_linewidth(0.6)
        cell.PAD = 0.03
        if r == 0:
            cell.set_facecolor(HEADER_BG)
            cell.set_text_props(fontweight='bold', color=INK, ha='left', va='center', fontsize=10.0)
        else:
            cell.set_facecolor(ROW_ALT if r % 2 == 0 else 'white')
            cell.set_text_props(color=INK, ha='left', va='center', fontsize=9.5)

    ax.text(0.0, -0.035, '* Testing N reflects longitudinal cohort (N=200); sub-sample (n=20) completed usability testing.',
            transform=ax.transAxes, fontsize=8.8, color=T_GREY, style='italic', ha='left', va='top')

    fig.subplots_adjust(left=0.01, right=0.99, top=0.92, bottom=0.08)
    _suptitle(fig, 'Table 1 | Characteristics of included studies', y_pos=0.97)
    _save(fig, 'table-1-study-characteristics')

make_table_1_characteristics()

## Figure S1 — Publication timeline and geographic distribution

Supplementary. Publication counts by year and by country. The final year is partial — `SEARCH_CLOSE_MONTHS = 1.5` scales its bar to the 16 February 2026 search date, so it is not comparable with the full years beside it. Peak and partial years are identified from the data rather than named.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE S1 (TIMELINE AND GEOGRAPHY)
# ═══════════════════════════════════════════════════════════════════════════════
def make_figure_s1_timeline_geography():
    print("⏳ Generating Figure S1 (timeline and geography)...")
    by_year  = CODED['year'].value_counts()
    years    = list(range(int(CODED['year'].min()), int(CODED['year'].max()) + 1))
    n_papers = [int(by_year.get(y, 0)) for y in years]
    # The final year is partial: the search closed on 16 February 2026.
    SEARCH_CLOSE_MONTHS = 1.5
    months   = [12] * (len(years) - 1) + [SEARCH_CLOSE_MONTHS]
    rate     = [n / m for n, m in zip(n_papers, months)]
    peak_year, part_year = years[int(np.argmax(n_papers))], years[-1]

    geo = CODED['country'].replace({'The Netherlands': 'Netherlands'}).value_counts()
    geo = geo.sort_values(ascending=False, kind='stable')
    countries, geo_n = list(geo.index), [int(v) for v in geo.values]

    # Adjusted figsize to be slightly taller (5.0 instead of 4.8) for title breathing room
    fig, (ax_t, ax_g) = plt.subplots(
        1, 2, figsize=(10.0, 5.0),
        gridspec_kw={'width_ratios': [1.0, 1.15], 'wspace': 0.38}
    )

    bar_colors = [BAR_PEAK if y == peak_year else BAR_PART if y == part_year else BAR_MAIN for y in years]
    x = np.arange(len(years))
    vbars = ax_t.bar(x, rate, color=bar_colors, width=0.40, zorder=3, edgecolor='white', linewidth=0.6)
    vbars[-1].set_hatch('////')
    vbars[-1].set_edgecolor(SPINE_COL)

    for bar, n, r in zip(vbars, n_papers, rate):
        if n == 0:
            ax_t.text(bar.get_x() + bar.get_width() / 2, 0.03, '0', ha='center', va='bottom', fontsize=10, color=SPINE_COL)
            continue
        ax_t.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.03, f'n = {n}', ha='center', va='bottom', fontsize=10.5, fontweight='bold', color=INK)
        ax_t.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.12, f'({r:.2f}/mo)', ha='center', va='bottom', fontsize=9, color=T_GREY, style='italic')

    ax_t.set_facecolor(BG_AX)
    ax_t.set_xticks(x)
    ax_t.set_xticklabels([str(y) for y in years], fontsize=FS_AX)
    ax_t.set_xlabel('Publication Year', fontsize=FS_LAB, color=INK, labelpad=5)
    ax_t.set_ylabel('Studies per Month', fontsize=FS_LAB, color=INK, labelpad=5)

    # Reduced max Y-limit so the bars look proportionally taller, matching the reference image
    ax_t.set_ylim(0, 1.95)
    ax_t.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
    for spine in ['top', 'right', 'left']: ax_t.spines[spine].set_visible(False)
    ax_t.spines['bottom'].set_color(SPINE_COL)
    ax_t.grid(axis='y', linestyle=':', alpha=0.55, color=GRID_COL)
    _panel_title(ax_t, 'A', 'Publication Timeline')

    # Updated legend labels to strictly match your correct image
    leg = [mpatches.Patch(facecolor=BAR_MAIN, label='Full calendar year'),
           mpatches.Patch(facecolor=BAR_PEAK, label=f'Peak year ({peak_year})'),
           mpatches.Patch(facecolor=BAR_PART, edgecolor=SPINE_COL, hatch='////', label=f'Partial year (Jan-Feb {part_year})')]
    ax_t.legend(handles=leg, loc='upper left', frameon=True, fontsize=9)

    _horiz_bar(ax_g, countries, geo_n, 'B', 'Geographic Distribution', colors=[BAR_MAIN]*len(geo_n), xpad=2.5)

    # FIXED OVERLAPPING TITLES:
    # Lowered the 'top' margin to 0.75 so the panel titles don't crash into the suptitle
    fig.subplots_adjust(left=0.07, right=0.97, top=0.75, bottom=0.14)
    _suptitle(fig, 'Publication Timeline and Geographic Distribution', y_pos=0.96)

    _save(fig, 'figure-s1-timeline-geography')

make_figure_s1_timeline_geography()

## Figure S2 — Explanation modality, scope and delivery format

Supplementary. How explanations were presented, whether they explained a single prediction or the whole model, and the platform used to deliver them.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE S2 (EXPLANATION MODALITY, SCOPE, DELIVERY)
# ═══════════════════════════════════════════════════════════════════════════════
def make_figure_s2_explanation_delivery():
    print("⏳ Generating Figure S2 (explanation modality, scope, delivery)...")
    modalities, modal_n  = grouped_counts('modality')
    scopes,     scope_n  = grouped_counts('scope')
    formats,    format_n = grouped_counts('delivery', wrap=True)
    fig, axes = plt.subplots(1, 3, figsize=(15.0, 5.6), gridspec_kw={'wspace': 0.38})
    _horiz_bar(axes[0], modalities, modal_n, 'A', 'Explanation Modality', colors=[BAR_MAIN]*len(modal_n), xpad=4.5)
    _horiz_bar(axes[1], scopes, scope_n, 'B', 'Explanation Scope', colors=[BAR_MAIN]*len(scope_n), xpad=4.5)
    _horiz_bar(axes[2], formats, format_n, 'C', 'Delivery Format', colors=[BAR_MAIN]*len(format_n), xpad=4.5)
    fig.subplots_adjust(left=0.06, right=0.98, top=0.68, bottom=0.15, wspace=0.38)
    _suptitle(fig, 'Explanation Modality, Scope, and Delivery Format', y_pos=0.92)
    _save(fig, 'figure-s2-explanation-delivery')

make_figure_s2_explanation_delivery()

## Unused — Patient involvement phase

**Not in the manuscript or supplement.** A standalone view of Figure 2, panel E, kept from the original script. Both now read the same derived counts, so they can no longer disagree. Its output is named `unused-` rather than given a figure number; delete this cell if it is not going to be used.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# UNUSED PANEL (PATIENT INVOLVEMENT PHASE)
# ═══════════════════════════════════════════════════════════════════════════════
def make_unused_patient_involvement():
    print("⏳ Generating unused panel (patient involvement phase)...")
    phases, counts = grouped_counts('phase')
    pcts = [c / N * 100 for c in counts]
    fig, ax = plt.subplots(figsize=(14.0, 5.4))
    ax.set_facecolor(BG_AX)
    y = np.arange(len(phases))
    bars = ax.barh(y, counts, color=BAR_MAIN, height=0.62, zorder=3, edgecolor='white', linewidth=0.5)
    for bar, v, p in zip(bars, counts, pcts):
        w = bar.get_width()
        ax.annotate(f'n = {v}   ({p:.0f}%)', xy=(w, bar.get_y() + bar.get_height() / 2), xytext=(8, 0), textcoords='offset points', va='center', ha='left', fontsize=FS_VAL, fontweight='bold', color=INK)
    ax.set_yticks(y); ax.set_yticklabels(phases, fontsize=FS_AX, color=INK); ax.invert_yaxis()
    ax.set_xlabel('Number of Studies', fontsize=FS_LAB, color=INK, labelpad=5)
    ax.set_xlim(0, max(counts) + 2.5)
    for spine in ['top', 'right']: ax.spines[spine].set_visible(False)
    ax.spines['left'].set_color(SPINE_COL); ax.spines['bottom'].set_color(SPINE_COL)
    ax.grid(axis='x', linestyle=':', alpha=0.55, color=GRID_COL)
    fig.subplots_adjust(left=0.14, right=0.98, top=0.78, bottom=0.15)
    _suptitle(fig, 'Patient Involvement Phase Across Included Studies', y_pos=0.93)
    _save(fig, 'unused-patient-involvement-phase')

make_unused_patient_involvement()

## Tables

Writes the manuscript and supplement tables into `tables/` as CSV, from the same derived frame the figures use.

- `table-1-study-characteristics.csv` — Table 1, one row per study.
- `summary-variable-frequencies.csv` — n and % for every grouped variable, i.e. the counts behind Figures 2, 3, S1 and S2 and quoted throughout the Results.
- `summary-trust-calibration.csv` — Figure 4 as a cross-tab, naming the studies in each cell.

Appendix D gets no file here. At one row per study across 24 mostly free-text variables it is not a printable table, so the supplement is right to link it: the workbook in `data/` is the appendix, and its CSV twin is already regenerated on every run.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TABLES — CSV for the manuscript and supplement
# ═══════════════════════════════════════════════════════════════════════════════
# Written from the same CODED frame as the figures, so the two cannot disagree.
# Appendix D is not written here: the extraction matrix in data/ is that appendix,
# and the supplement links to it on OSF rather than printing 24 free-text columns.

TABLES_DIR = Path('tables')
TABLES_DIR.mkdir(exist_ok=True)

def _write(frame, filename):
    path = TABLES_DIR / filename
    frame.to_csv(path, index=False)
    print(f'   ✅ Saved: {path}  ({len(frame)} rows)')

# Table 1 — characteristics of included studies.
_write(table1(), 'table-1-study-characteristics.csv')

# Counts behind Figures 2, 3, S1 and S2, as reported in the Results.
frequencies = pd.DataFrame(
    [{'Variable': label, 'Category': category, 'Studies (n)': n,
      'Percentage': round(n / N * 100, 1)}
     for key, label in VARIABLE_LABELS.items()
     for category, n in zip(*grouped_counts(key))])
_write(frequencies, 'summary-variable-frequencies.csv')

# Figure 4 in tabular form, listing which studies fall in each cell.
calibration = pd.DataFrame(
    [{'Patient trust': level.title(), 'Model performance': performance.title(),
      'Matrix cell': TRUST_CELL_NAMES[(level, performance)],
      'Classification': TRUST_CELL_KIND[(level, performance)].title(),
      'Studies (n)': len(members), 'Percentage': round(len(members) / N * 100, 1),
      'Studies': '; '.join(members)}
     for (level, performance), members in
     ((key, trust_members(key)) for key in TRUST_CELL_NAMES)])
_write(calibration, 'summary-trust-calibration.csv')